## trees

#### decision tree

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


In [3]:
data = pd.read_csv('../data/train/train.csv')
data_test = pd.read_csv('../data/test/test.csv')

In [102]:
data['CabinKnown'] = data['Cabin'].notna()
data['Age'] = data['Age'].fillna(data['Age'].median())
data['Embarked'] = data['Embarked'].fillna('S')

data_procced_categories = data.copy()
data_procced_categories['CabinKnown'] = data['CabinKnown'].astype('int32')
replace_dict = {'male': 0, 'female': 1}
data_procced_categories['Sex'] = data_procced_categories['Sex'].replace(replace_dict)
data_procced_categories.head()



,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,CabinKnown
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,NaN,S,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,C85,C,1
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,C123,S,1
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,NaN,S,0


In [104]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')

# data_procced_categories['Embarked'] = encoder.fit_transform(data_procced_categories['Embarked'])
encoded = encoder.fit_transform(data_procced_categories[['Embarked']],)
encoded_df= pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(["Embarked"]),
    index=data_procced_categories.index,
)

data_procced_categories = pd.concat(
    [
        data_procced_categories.drop(columns=['Embarked']),
        encoded_df
    ],
    axis=1
)
data_procced_categories


KeyError: "None of [Index(['Embarked'], dtype='str')] are in the [columns]"

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, GridSearchCV



jopa = data_procced_categories.copy()
# jopa[features_to_scale] = scaler.fit_transform(data[features_to_scale])
train_X = jopa.copy()
train_X = train_X.drop(columns=['Survived', 'Name', 'Ticket', 'Cabin', 'PassengerId'])
train_y = data['Survived']

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_train_metric = []
fold_val_metric = []

param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [None, 3, 5, 7, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

dtree = DecisionTreeClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=dtree, param_grid=param_grid, cv=skf, scoring='accuracy', n_jobs=1, return_train_score=True
)
grid_search.fit(train_X, train_y)

print(grid_search.best_estimator_)
print(f'Best params: {grid_search.best_params_}')
print(f'Best cross val score: {grid_search.best_score_:4f}')

# for train_indices, val_indices in skf.split(train_X, train_y):
#     x_train_fold = train_X.iloc[train_indices]
#     x_val_fold = train_X.iloc[val_indices]

#     y_train_fold = train_y.iloc[train_indices]
#     y_val_fold = train_y.iloc[val_indices]
#     clf = DecisionTreeClassifier(max_depth=3,
#                                   random_state=42,
#                                   criterion='entropy',
#                                   min_samples_leaf=2,
#                                   )
#     clf.fit(x_train_fold, y_train_fold)
#     y_train_pred = clf.predict(x_train_fold)
#     y_pred = clf.predict(x_val_fold)

#     accuracy_train = accuracy_score(y_train_fold, y_train_pred)
#     accuracy = accuracy_score(y_val_fold, y_pred)
#     fold_val_metric.append(accuracy)
#     fold_train_metric.append(accuracy_train)

# print(f"Mean val accuracy: {sum(fold_val_metric) / len(fold_val_metric):.4f}")
# print(f"Mean train accuracy: {sum(fold_train_metric) / len(fold_train_metric):.4f}")
# print(fold_val_metric)


# plt.figure(figsize=(12, 8))
# plot_tree(clf, filled=True)
# plt.show()

# Best params: {'criterion': 'entropy', 'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 2}
# Best cross val score: 0.812573
#                       0.821549



DecisionTreeClassifier(max_depth=10, min_samples_split=10, random_state=42)
Best params: {'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 10}
Best cross val score: 0.821549


In [118]:
from sklearn.ensemble import RandomForestClassifier

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, GridSearchCV, RandomizedSearchCV



jopa = data_procced_categories.copy()
train_X = jopa.copy()
train_X = train_X.drop(columns=['Survived', 'Name', 'Ticket', 'Cabin', 'PassengerId'])
train_y = data['Survived']

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_train_metric = []
fold_val_metric = []

param_grid = {
    "max_depth": [5, 7],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "bootstrap": [True, False]
}

param_distributions = {
    "n_estimators": [100, 200, 300, 500, 800],
    "criterion": ["gini", "entropy", "log_loss"],
    "max_depth": [None, 3, 5, 7, 10, 15, 20],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True, False],
}

dtree = DecisionTreeClassifier(random_state=42)
clf = RandomForestClassifier(random_state=42, criterion='entropy', n_estimators=200, bootstrap=True)

# TODO: TRY RANDOMSEARCHCV 
grid_search = GridSearchCV(
    estimator=clf, param_grid=param_grid, cv=skf, scoring='accuracy', n_jobs=-1, return_train_score=True, verbose=2
)
random_grid_search = RandomizedSearchCV(
    estimator=clf,
    scoring='accuracy',
    param_distributions=param_distributions,
    n_iter=50,
    verbose=2
    )
grid_search.fit(train_X, train_y)

print(grid_search.best_estimator_)
print(f'Best params: {grid_search.best_params_}')
print(f'Best cross val score: {grid_search.best_score_:4f}')


# plt.figure(figsize=(12, 8))
# plot_tree(clf, filled=True)
# plt.show()

# FIXME: BEST BEST
# RandomForestClassifier(criterion='entropy', max_depth=5, n_estimators=200,
#                        random_state=42)
# Best params: {'criterion': 'entropy', 'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
# Best cross val score: 0.833865



Fitting 5 folds for each of 16 candidates, totalling 80 fits
[CV] END bootstrap=True, max_depth=5, min_samples_leaf=1, min_samples_split=2; total time=   0.2s
[CV] END bootstrap=True, max_depth=5, min_samples_leaf=1, min_samples_split=2; total time=   0.2s
[CV] END bootstrap=True, max_depth=5, min_samples_leaf=1, min_samples_split=2; total time=   0.3s
[CV] END bootstrap=True, max_depth=5, min_samples_leaf=1, min_samples_split=5; total time=   0.2s
[CV] END bootstrap=True, max_depth=5, min_samples_leaf=1, min_samples_split=2; total time=   0.3s
[CV] END bootstrap=True, max_depth=5, min_samples_leaf=1, min_samples_split=2; total time=   0.3s
[CV] END bootstrap=True, max_depth=5, min_samples_leaf=1, min_samples_split=5; total time=   0.3s
[CV] END bootstrap=True, max_depth=5, min_samples_leaf=1, min_samples_split=5; total time=   0.2s
[CV] END bootstrap=True, max_depth=5, min_samples_leaf=1, min_samples_split=5; total time=   0.2s
[CV] END bootstrap=True, max_depth=5, min_samples_leaf=1,